In [ ]:
from pathlib import Path

import numpy as np
from careamics_portfolio import PortfolioManager
from matplotlib import pyplot as plt


In [ ]:

root = Path("data")
root.mkdir(exist_ok=True, parents=True)

files = PortfolioManager().denoiseg.DSB2018_n0.download(root)

In [ ]:
test = np.load(files[-2], allow_pickle=True)["X_test"]
test_tar = np.load(files[-2], allow_pickle=True)["Y_test"]
train = np.load(files[-1], allow_pickle=True)["X_train"]
train_tar = np.load(files[-1], allow_pickle=True)["Y_train"]
val = np.load(files[-1], allow_pickle=True)["X_val"]
val_tar = np.load(files[-1], allow_pickle=True)["Y_val"]


test = list(test)
test_tar =list(test_tar)

print(
    f"Train size: {len(train)}\n"
    f"Val size: {len(val)}\n"
    f"Test size: {len(test)}\n"
)

In [ ]:
# show images
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
axes[0, 0].imshow(train[150])
axes[1, 0].imshow(train_tar[150])

axes[0, 1].imshow(val[100])
axes[1, 1].imshow(val_tar[100])

axes[0, 2].imshow(test[25])
axes[1, 2].imshow(test_tar[25])
plt.tight_layout()
plt.show()

In [ ]:
single_class_path = root / "single"
single_class_path.mkdir(parents=True, exist_ok=True)
multi_class_path = root / "multi"
multi_class_path.mkdir(parents=True, exist_ok=True)

print(single_class_path.parent)

## Single class

In [ ]:
sin_train_tar = train_tar > 0
sin_val_tar = val_tar > 0
sin_test_tar = [t > 0 for t in test_tar]

# show images
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
axes[0, 0].imshow(train[150])
axes[1, 0].imshow(sin_train_tar[150])

axes[0, 1].imshow(val[100])
axes[1, 1].imshow(sin_val_tar[100])

axes[0, 2].imshow(test[25])
axes[1, 2].imshow(sin_test_tar[25])
plt.tight_layout()
plt.show()

In [ ]:
# save
np.save(single_class_path / "train.npy", train)
np.save(single_class_path / "train_target.npy", sin_train_tar)
np.save(single_class_path / "val.npy", val)
np.save(single_class_path / "val_target.npy", sin_val_tar)

(single_class_path / "test").mkdir(parents=True, exist_ok=True)
(single_class_path / "test_target").mkdir(parents=True, exist_ok=True)

for i in range(len(test)):
    np.save(single_class_path / "test" / f"test_{i}.npy", test[i])
    np.save(single_class_path / "test_target" / f"test_{i}.npy", sin_test_tar[i])

## Multi class

In [ ]:
from skimage.morphology import dilation, disk
from skimage.segmentation import find_boundaries


def get_multimask(instance, binary):
    """Generate a multiclass target from the binary and instance maps."""
    borders = find_boundaries(instance, mode="inner")
    borders = dilation(borders, disk(2))  # increase thickness
    multiclass = binary.astype(np.uint8).copy()
    multiclass[borders & (binary > 0)] = 2

    return multiclass


mult_train_tar = np.array([
    get_multimask(train_tar[i], sin_train_tar[i])
    for i in range(len(train_tar))
])
mult_val_tar = np.array([
    get_multimask(val_tar[i], sin_val_tar[i])
    for i in range(len(val_tar))
])
mult_test_tar = [
    get_multimask(test_tar[i], sin_test_tar[i])
    for i in range(len(test_tar))
]


# show images
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
axes[0, 0].imshow(train[150])
axes[1, 0].imshow(mult_train_tar[150])

axes[0, 1].imshow(val[100])
axes[1, 1].imshow(mult_val_tar[100])

axes[0, 2].imshow(test[25])
axes[1, 2].imshow(mult_test_tar[25])
plt.tight_layout()
plt.show()

In [ ]:
def to_one_hot(labels):
    """Generate one-hot encoded labels from class labels."""
    one_hot = np.eye(3, dtype=np.uint8)[labels]
    one_hot_chw = np.moveaxis(one_hot, -1, 0)
    return one_hot_chw


# show images
fig, axes = plt.subplots(3, 3, figsize=(10, 6))
axes[0, 0].imshow(train[150])
axes[1, 0].imshow(to_one_hot(mult_train_tar[150])[1])
axes[2, 0].imshow(to_one_hot(mult_train_tar[150])[2])

axes[0, 1].imshow(val[100])
axes[1, 1].imshow(to_one_hot(mult_val_tar[100])[1])
axes[2, 1].imshow(to_one_hot(mult_val_tar[100])[2])

axes[0, 2].imshow(test[25])
axes[1, 2].imshow(to_one_hot(mult_test_tar[25])[1])
axes[2, 2].imshow(to_one_hot(mult_test_tar[25])[2])
plt.tight_layout()
plt.show()


In [ ]:

# save
np.save(multi_class_path / "train", train)
np.save(multi_class_path / "train_target", mult_train_tar)
np.save(multi_class_path / "val", val)
np.save(multi_class_path / "val_target", mult_val_tar)

(multi_class_path / "test").mkdir(parents=True, exist_ok=True)
(multi_class_path / "test_target").mkdir(parents=True, exist_ok=True)

for i in range(len(test)):
    np.save(multi_class_path / "test" / f"test_{i}", test[i])
    np.save(multi_class_path / "test_target" / f"test_{i}", mult_test_tar[i])